# MLflow Prompt Registry
In this notebook, we will demonstrate how to make use of MLflow's prompt registry functionality. We'll move from simple prompt registration to versioned prompt configuration, loading prompts for inference, and automatically rewriting prompts from example behavior.

## 0: Notebook Setup
We'll set ourselves up for success by importing our Python dependencies and setting up the MLflow connection. This setup also defines the prompt templates, sample content, and labeled sentiment examples that the rest of the notebook will reuse.

In [1]:
# Importing the necessary Python libraries
import json
import mlflow
from mlflow.genai.datasets import create_dataset
from mlflow.genai.optimize.optimizers import GepaPromptOptimizer
from mlflow.genai.scorers import Equivalence
from openai import OpenAI
from pydantic import BaseModel, Field

/Users/dkhundley/Documents/Repositories/mlflow-tutorial/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting the base URL for our MLflow instance
MLFLOW_BASE_URL = 'http://127.0.0.1:5000'

# Setting the OpenAI 5.4 nano endpoint from MLflow
OPENAI_GPT_5_4_NANO_ENDPOINT = 'openai-gpt-5.4-nano'

# Instantiating the OpenAI SDK client
openai_client = OpenAI(
    base_url = f'{MLFLOW_BASE_URL}/gateway/openai/v1',
    api_key = 'dummy'
)

# Pointing to the MLflow tracking server
mlflow.set_tracking_uri(MLFLOW_BASE_URL)


In [3]:
# Setting our summarization text prompt
SUMMARIZATION_PROMPT = '''
Summarize the following text in {{ num_sentences }}.

Here is the text to summarize:
{{ text }}
'''

# Setting our starter chat prompt
SUMMARIZATION_CHAT_PROMPT = [
    {
        'role': 'system',
        'content': 'Summarize content you are provided with in {{ num_sentences }} sentences.'
    },
    {
        'role': 'user',
        'content': 'Here is the text to summarize: {{ sentences }}'
    }
]

# Creating a dummy write up of MLflow to later summarize
MLFLOW_WRITE_UP = '''
MLflow is an open source platform for managing the machine learning and generative AI lifecycle. It helps teams organize experiments, track model behavior, package code, register models, and move machine learning systems from development into production. Instead of relying on scattered notebooks, ad hoc files, and manual handoffs, MLflow provides a common system of record for models, prompts, evaluations, and deployment artifacts.

One of MLflow’s core strengths is experiment tracking. Data scientists and machine learning engineers can log parameters, metrics, artifacts, datasets, and model outputs during training or evaluation runs. This makes it easier to compare different approaches, reproduce past results, and understand why one model performed better than another. The MLflow UI provides a central place to inspect runs, compare metrics, review artifacts, and share results with other team members.

MLflow also supports model packaging and model management. With MLflow Models, teams can save models in a standardized format that includes the model artifact, environment information, and inference interface. This makes it easier to move a model between local development, batch scoring, real-time serving, and other deployment targets. The MLflow Model Registry adds governance by allowing teams to register versions of a model, track its stage, document changes, and coordinate promotion toward production.

For generative AI use cases, MLflow has expanded beyond traditional model tracking. Teams can use MLflow to track prompts, evaluate LLM responses, capture traces, compare different providers or models, and monitor how applications behave across development and production workflows. This is especially useful when building applications that depend on prompt templates, retrieval systems, agents, or external model APIs. By recording inputs, outputs, metadata, and evaluation results, MLflow helps teams understand not only whether an application works, but why it works.

MLflow is valuable because it brings structure and repeatability to work that can otherwise become difficult to manage. Machine learning and AI projects often involve many experiments, changing datasets, multiple contributors, and evolving deployment requirements. MLflow gives teams a shared framework for tracking decisions, reviewing results, and improving systems over time. Whether a team is training a classical machine learning model, evaluating a large language model, or building a GenAI application, MLflow helps make the process more observable, reproducible, and collaborative.
'''

# Creating a dummy sentiment analysis prompt
SENTIMENT_ANALYSIS_PROMPT = '''
Yo check out these sentences. Determine the sentiment. You got 3 choices: positive, neutral, or negative.

Have at it:

{{ sentences }}
'''

# Creating sentiment examples
SENTIMENT_EXAMPLES = [
    {
      "text": "MLflow made it so much easier to keep track of all my machine learning experiments.",
      "sentiment": "positive"
    },
    {
      "text": "I love how MLflow lets me compare model runs without digging through old notebooks.",
      "sentiment": "positive"
    },
    {
      "text": "Registering models in MLflow has made deployments much more organized.",
      "sentiment": "positive"
    },
    {
      "text": "Our team adopted MLflow, and collaboration has improved dramatically.",
      "sentiment": "positive"
    },
    {
      "text": "The MLflow UI is one of my favorite ways to inspect experiment results.",
      "sentiment": "positive"
    },
    {
      "text": "I'm impressed by how easy it is to log metrics and artifacts with MLflow.",
      "sentiment": "positive"
    },
    {
      "text": "MLflow saved me hours of work when I needed to reproduce an old experiment.",
      "sentiment": "positive"
    },
    {
      "text": "Grrr... I really don't like when people refuse to use MLflow.",
      "sentiment": "negative"
    },
    {
      "text": "I accidentally deleted my MLflow experiment, and now I'm frustrated.",
      "sentiment": "negative"
    },
    {
      "text": "I can't believe someone stored all their experiment results in random spreadsheets instead of MLflow.",
      "sentiment": "negative"
    },
    {
      "text": "Our MLflow server was offline this morning, which completely interrupted my workflow.",
      "sentiment": "negative"
    },
    {
      "text": "I forgot to log my model to MLflow and now I have to rerun everything.",
      "sentiment": "negative"
    },
    {
      "text": "Debugging a misconfigured MLflow Tracking Server was not a fun afternoon.",
      "sentiment": "negative"
    },
    {
      "text": "MLflow is an open-source platform for managing machine learning workflows.",
      "sentiment": "neutral"
    },
    {
      "text": "Our organization uses MLflow to track experiments.",
      "sentiment": "neutral"
    },
    {
      "text": "The latest training run was logged to MLflow.",
      "sentiment": "neutral"
    },
    {
      "text": "MLflow supports experiment tracking, model management, and evaluation features.",
      "sentiment": "neutral"
    },
    {
      "text": "The data science team reviewed the metrics stored in MLflow during today's meeting.",
      "sentiment": "neutral"
    },
    {
      "text": "An MLflow Tracking Server is running in our development environment.",
      "sentiment": "neutral"
    },
    {
      "text": "The engineer opened the MLflow UI to inspect the latest experiment results.",
      "sentiment": "neutral"
    }
]

## 1: Basic Prompt Registration
In this section, we'll demonstrate how you can simply register a prompt to the MLflow prompt registry. Before we get into this, it is first important to recognize that MLflow supports two different prompt types:

1. **Text**: This is a simple string. It accepts variables delineated by double curly braces (e.g. `{{ variable_name }}`)
2. **Chat**: This simulates the beginning of a back-and-forth chat conversation. This may be helpful for things like few shot prompting.

In the cells below, we will demonstrate how you can simply register each of these respective types of prompts programatically.

### 1.1: Registering a Text Prompt
We'll start by registering a plain text prompt template. This creates a named prompt version in MLflow that can be loaded and reused later.

In [4]:
# Registering our summarization prompt as a text prompt
registered_summarization_prompt = mlflow.genai.register_prompt(
    name = 'summarization-prompt',
    template = SUMMARIZATION_PROMPT,
    commit_message = 'Registering the initial version of the summarization prompt'
)

print(f"Created prompt '{registered_summarization_prompt.name}' (version {registered_summarization_prompt.version})")

2026/07/04 09:05:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-prompt, version 14


Created prompt 'summarization-prompt' (version 14)


### 1.2: Registering a Chat Prompt
Next, we'll register a chat-style prompt made up of role-based messages. This is useful when you want the prompt registry to preserve a conversational structure rather than a single text block.

In [5]:
# Registering our summarization prompt as a chat prompt
registered_summarization_chat_prompt = mlflow.genai.register_prompt(
    name = 'summarization-chat-prompt',
    template = SUMMARIZATION_CHAT_PROMPT,
    commit_message = 'Registering the initial version of the summarization chat prompt'
)

print(f"Created prompt '{registered_summarization_chat_prompt.name}' (version {registered_summarization_chat_prompt.version})")

2026/07/04 09:05:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-chat-prompt, version 13


Created prompt 'summarization-chat-prompt' (version 13)


## 2: Advanced Prompt Registration
In the previous section, we registered a basic text and chat version of our summarization prompt. In this section, we'll take things a step further by demonstrating how we can apply additional configuration to each of these prompts. We'll overwrite these prompts in MLflow's prompt registry to also demonstrate how versioning works.

We'll be adding the following bits of information:

1. **Response format**: You can set the expected response format either using Pydantic or JSON. In the cell below, we will set a response format using Pydantic.
2. **Model configuration**: It is possible to set the model configuration that the prompt is intended to be used with. For our purposes, we will be making use of the OpenAI GPT-5.4 nano endpoint as we have manifested through MLflow's AI Gateway.
3. **Tags**: These are simply additional bits of metadata about the registered prompt.

In [6]:
# Creating a response structure using Pydantic
class SummaryResponseFormat(BaseModel):
    summary: str = Field(..., description = 'Summary of the content')

### 2.1: Registering a Text Prompt the Advanced Way
Here, we'll re-register the text prompt with additional metadata and model configuration. Because the prompt name is the same, MLflow records this as a new version rather than replacing the old version in place.

In [7]:
# Re-registering the text prompt with additional configuration
registered_summarization_prompt = mlflow.genai.register_prompt(
    name = 'summarization-prompt',
    template = SUMMARIZATION_PROMPT,
    commit_message = 'Registering the prompt with additional config information',
    response_format = SummaryResponseFormat,
    model_config = {
        'model_name': OPENAI_GPT_5_4_NANO_ENDPOINT,
        'temperature': 0.7,
        'max_tokens': 1000
    },
    tags = {
        'author': 'dkhundley'
    }
)

print(f"Created prompt '{registered_summarization_prompt.name}' (version {registered_summarization_prompt.version})")

2026/07/04 09:05:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-prompt, version 15


Created prompt 'summarization-prompt' (version 15)


### 2.2: Registering a Chat Prompt the Advanced Way
We'll apply the same advanced registration pattern to the chat prompt. This keeps the chat template versioned while also attaching the response format, target model, and prompt tags.

In [8]:
# Re-registering the text prompt with additional configuration
registered_summarization_chat_prompt = mlflow.genai.register_prompt(
    name = 'summarization-chat-prompt',
    template = SUMMARIZATION_CHAT_PROMPT,
    commit_message = 'Registering the chat prompt with additional config information',
    response_format = SummaryResponseFormat,
    model_config = {
        'model_name': OPENAI_GPT_5_4_NANO_ENDPOINT,
        'temperature': 0.7,
        'max_tokens': 1000
    },
    tags = {
        'author': 'dkhundley'
    }
)

print(f"Created prompt '{registered_summarization_chat_prompt.name}' (version {registered_summarization_chat_prompt.version})")

2026/07/04 09:05:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-chat-prompt, version 14


Created prompt 'summarization-chat-prompt' (version 14)


## 3: Loading Prompts from the Prompt Registry
Now that the prompts are registered, we'll load them back from MLflow and use their stored configuration during inference. This demonstrates why registry metadata is helpful: the prompt, model name, and response format can travel together.

### 3.1: Using the Text Prompt
In this example, we'll load the text prompt, format its variables, and send the formatted prompt to the configured model. The response format stored with the prompt is used to request structured JSON output.

In [9]:
# Loading the summarization text prompt from MLflow
loaded_summarization_prompt = mlflow.genai.load_prompt('summarization-prompt')

# Formatting the prompt with number of sentences and text to summarize
formatted_summarization_prompt = loaded_summarization_prompt.format(num_sentences = 5, text = MLFLOW_WRITE_UP)

# Invoking the OpenAI model with the OpenAI SDK
response = openai_client.chat.completions.create(
    model = loaded_summarization_prompt.model_config['model_name'],
    messages = [
        {
            'role': 'user',
            'content': formatted_summarization_prompt
        }
    ],
    response_format = {
        'type': 'json_schema',
        'json_schema': {
            'name': 'summary_response',
            'schema': loaded_summarization_prompt.response_format
        }
    }
)

# Printing the response
print(response.choices[0].message.content)


{"summary":"MLflow is an open-source platform that helps teams manage the full machine learning and generative AI lifecycle—from organizing experiments and tracking metrics and artifacts, to packaging models and registering versions for production. It provides governance with a model registry and extends tracking to GenAI by logging prompts, evaluating LLM outputs, capturing traces, and monitoring application behavior. Overall, MLflow adds structure, repeatability, and collaboration to complex, evolving AI projects."}


### 3.2: Using the Chat Prompt
This example follows the same loading pattern with the chat prompt. The goal is to show that prompt registry entries can be treated as reusable inference assets, regardless of whether they began as text or chat templates.

In [10]:
# Loading the summarization chat prompt from MLflow
loaded_summarization_chat_prompt = mlflow.genai.load_prompt('summarization-prompt')

# Formatting the prompt with number of sentences and text to summarize
formatted_summarization_chat_prompt = loaded_summarization_chat_prompt.format(num_sentences = 5, text = MLFLOW_WRITE_UP)

# Invoking the OpenAI model with the OpenAI SDK
response = openai_client.chat.completions.create(
    model = loaded_summarization_chat_prompt.model_config['model_name'],
    messages = [
        {
            'role': 'user',
            'content': formatted_summarization_chat_prompt
        }
    ],
    response_format = {
        'type': 'json_schema',
        'json_schema': {
            'name': 'summary_response',
            'schema': loaded_summarization_chat_prompt.response_format
        }
    }
)

# Printing the response
print(response.choices[0].message.content)


{"summary":"MLflow is an open-source platform that manages the full machine learning and generative AI lifecycle, replacing scattered notebooks with a shared system of record for models, prompts, evaluations, and deployment artifacts. It provides strong experiment tracking to log parameters, metrics, and artifacts for easy comparison and reproducibility. MLflow also supports standardized model packaging and a registry for versioning and governance. For GenAI, it helps track prompts and LLM responses, evaluate outputs, and monitor application behavior across dev and production. Overall, MLflow adds structure, repeatability, and collaboration to evolving AI projects."}


## 4: Auto-rewrite of Prompts
In this section, we'll demonstrate how MLflow can automatically rewrite a prompt by using examples of the behavior we want the prompt to produce. Since we already have labeled sentiment examples in the notebook setup, we'll skip the output capture step and use those examples directly as our rewrite dataset. The result will be saved as a new version of the same registered prompt, preserving the original prompt history.

### 4.1: Registering the Original Sentiment Prompt
We'll first register the intentionally rough sentiment prompt as the baseline version. (Random aside: I had ChatGPT help me write this markdown cell, but I wrote that "intentionally rough" prompt. Thank you, ChatGPT, for recognizing that is "intentionally rough". 😂)

In [11]:
# Registering our starter sentiment analysis prompt
registered_sentiment_prompt = mlflow.genai.register_prompt(
    name = 'sentiment-analysis-prompt',
    template = SENTIMENT_ANALYSIS_PROMPT,
    commit_message = 'Registering the initial sentiment analysis prompt',
    model_config = {
        'model_name': OPENAI_GPT_5_4_NANO_ENDPOINT,
        'temperature': 0
    },
    tags = {
        'task': 'sentiment-analysis',
    }
)

print(f"Created prompt '{registered_sentiment_prompt.name}' (version {registered_sentiment_prompt.version})")

2026/07/04 09:05:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: sentiment-analysis-prompt, version 11


Created prompt 'sentiment-analysis-prompt' (version 11)


### 4.2: Creating a Prompt Rewrite Dataset
Next, we'll convert the labeled sentiment examples into an MLflow GenAI evaluation dataset. Each record contains prompt inputs, the expected output label, and an explicit expectation field for the equivalence scorer.

In [12]:
# Creating records that map prompt inputs to expected outputs
sentiment_rewrite_records = [
    {
        'inputs': {
            'sentences': example['text']
        },
        'outputs': example['sentiment'],
        'expectations': {
            'expected_response': example['sentiment']
        }
    }
    for example in SENTIMENT_EXAMPLES
]

# Creating a dataset for prompt rewriting
sentiment_rewrite_dataset = create_dataset(
    name = 'sentiment-prompt-rewrite-dataset',
    tags = {
        'task': 'sentiment-analysis',
        'source': 'prompt-registry-notebook'
    }
)

# Adding the records to the dataset
sentiment_rewrite_dataset = sentiment_rewrite_dataset.merge_records(sentiment_rewrite_records)

print(f'Created dataset with {len(sentiment_rewrite_records)} records')

Created dataset with 20 records


### 4.3: Defining the Sentiment Prediction Function
The optimizer needs a prediction function that shows how the prompt is used in the real application path. This function loads the registered prompt, formats it with a sentence, calls the MLflow Gateway endpoint, and returns the model's sentiment label.

In [13]:
# Defining a prediction function that uses the registered sentiment prompt
def predict_sentiment(sentences: str) -> str:
    loaded_sentiment_prompt = mlflow.genai.load_prompt(registered_sentiment_prompt.uri)
    formatted_sentiment_prompt = loaded_sentiment_prompt.format(sentences = sentences)

    response = openai_client.chat.completions.create(
        model = loaded_sentiment_prompt.model_config['model_name'],
        messages = [
            {
                'role': 'user',
                'content': formatted_sentiment_prompt
            }
        ],
        temperature = loaded_sentiment_prompt.model_config['temperature']
    )

    return response.choices[0].message.content.strip().lower()


### 4.4: Rewriting the Prompt
This is the core rewrite loop. GEPA is a prompt optimization algorithm that uses model-generated reflection to propose new prompt candidates, then keeps candidates that improve the measured task score. Rather than manually editing the prompt and guessing whether it is better, GEPA repeatedly evaluates, reflects, rewrites, and compares prompt versions against the examples we provide.

In this notebook, MLflow evaluates the current registered prompt against the sentiment dataset, GEPA proposes candidate prompt text based on examples and feedback, and the equivalence scorer checks whether each candidate better matches the expected labels. Because our LLM is exposed through MLflow Gateway, the custom proposer routes rewrite requests through the existing OpenAI-compatible gateway client instead of calling a provider directly.

When optimization finds the best candidate, MLflow automatically registers that rewritten template as a new version of `sentiment-analysis-prompt`, so the original and improved prompts remain comparable in the registry.

In [14]:
# Defining a GEPA candidate proposer that routes rewrite calls through MLflow Gateway
def propose_sentiment_prompt_rewrite(
    candidate: dict[str, str],
    reflective_dataset: dict[str, list[dict]],
    components_to_update: list[str]
) -> dict[str, str]:
    rewritten_candidate = candidate.copy()

    for component_name in components_to_update:
        rewrite_examples = reflective_dataset.get(component_name, [])[:8]
        rewrite_request = f'''
        Rewrite this sentiment analysis prompt to better match the expected labels.

        Keep the prompt as a text prompt that uses the variable {{{{ sentences }}}}.
        The only valid labels are positive, negative, and neutral.
        Return only the rewritten prompt template.

        Current prompt:
        {candidate[component_name]}

        Optimization examples and feedback:
        {json.dumps(rewrite_examples, indent = 2)}
        '''

        response = openai_client.chat.completions.create(
            model = OPENAI_GPT_5_4_NANO_ENDPOINT,
            messages = [
                {
                    'role': 'user',
                    'content': rewrite_request
                }
            ],
            temperature = 0.2
        )

        rewritten_candidate[component_name] = response.choices[0].message.content.strip()

    return rewritten_candidate

In [15]:
# Optimizing our summarization prompt
prompt_rewrite_result = mlflow.genai.optimize_prompts(
    predict_fn = predict_sentiment,
    train_data = sentiment_rewrite_dataset,
    prompt_uris = [registered_sentiment_prompt.uri],
    optimizer = GepaPromptOptimizer(
        reflection_model = f'gateway:/{OPENAI_GPT_5_4_NANO_ENDPOINT}',
        max_metric_calls = 60,
        gepa_kwargs = {
            'custom_candidate_proposer': propose_sentiment_prompt_rewrite
        }
    ),
    scorers = [
        Equivalence(model = f'gateway:/{OPENAI_GPT_5_4_NANO_ENDPOINT}')
    ]
)

# Extracting the rewritten summarization prompt
rewritten_sentiment_prompt = prompt_rewrite_result.optimized_prompts[0]

print(f'Original version: {registered_sentiment_prompt.version}')
print(f'Rewritten version: {rewritten_sentiment_prompt.version}')
print(f'Rewritten prompt URI: {rewritten_sentiment_prompt.uri}')
print(f'Initial score: {prompt_rewrite_result.initial_eval_score}')
print(f'Final score: {prompt_rewrite_result.final_eval_score}')
print('\nRewritten prompt template:')
print(rewritten_sentiment_prompt.template)

2026/07/04 09:05:40 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/07/04 09:05:40 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/Users/dkhundley/Documents/Repositories/mlflow-tutorial/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Iteration 0: Base program full valset score: 0.8 over 20 / 20 examples
Iteration 1: Selected program 0 score: 0.8
Iteration 1: Proposed new text for sentiment-analysis-prompt: Classify the sentiment of the text below. Choose exactly one label: positive, neutral, or negative.

Return only the label (no extra words).

Text:
{{ sentences }}
Iteration 1: New subsample score 2.0 is not better than old score 2.0, skipping
Iteration 2: Selected program 0 score: 0.8
Iteration 2: Proposed new text for sentiment-analysis-prompt: Classify the sentiment of the text below as exactly one of: positive, neutral, or negative.

Text:
{{ sentences }}

Answer with only the single label (positive, neutral, or negative).
Iteration 2: New subsample score 3.0 is better than old score 2.0. Continue to full eval and add to candidate pool.
Iteration 2: Found a better program on the valset with score 0.85.
Iteration 2: Valset score for new program: 0.85 (coverage 20 / 20)
Iteration 2: Val aggregate for new progra

2026/07/04 09:06:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: sentiment-analysis-prompt, version 12


Iteration 4: New subsample score 2.0 is not better than old score 2.0, skipping
🏃 View run stylish-gnat-137 at: http://127.0.0.1:5000/#/experiments/0/runs/6f5e7b0ff2ce4df282b67ef18307db24
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
Original version: 11
Rewritten version: 12
Rewritten prompt URI: prompts:/sentiment-analysis-prompt/12
Initial score: 0.8
Final score: 0.85

Rewritten prompt template:
Classify the sentiment of the text below as exactly one of: positive, neutral, or negative.

Text:
{{ sentences }}

Answer with only the single label (positive, neutral, or negative).


### 4.5: Loading the Rewritten Prompt
Finally, we'll load the optimized prompt version that `optimize_prompts()` already registered in MLflow and test it on a fresh sentence. This mirrors the earlier loading examples, but now the prompt being used is the optimized version returned by the rewrite workflow.

In [16]:
# Loading the rewritten prompt and testing it on a new sentence
loaded_rewritten_sentiment_prompt = mlflow.genai.load_prompt(rewritten_sentiment_prompt.uri)
formatted_rewritten_sentiment_prompt = loaded_rewritten_sentiment_prompt.format(
    sentences = 'MLflow tracing made it much simpler to understand what my GenAI app was doing.'
)

response = openai_client.chat.completions.create(
    model = loaded_rewritten_sentiment_prompt.model_config['model_name'],
    messages = [
        {
            'role': 'user',
            'content': formatted_rewritten_sentiment_prompt
        }
    ],
    temperature = loaded_rewritten_sentiment_prompt.model_config['temperature']
)

print(response.choices[0].message.content.strip())

positive
